# Chapter 30 – Worked examples

This notebook runs the examples of [Chapter 30](../index.md) live. The code lives in the modules next to it
(`two_libraries.py`, `incidence.py`, `sparse_vs_dense.py`, `ieee14_flows.py`, `case57_ac_dc.py`,
`loading_query.py`); the cells only call them, so a change in a module is a change here too.

Run it from this folder (the kernel's working directory must be `src/`), or open it in Jupyter Lab from the
repository.

In [ ]:
import case57_ac_dc
import ieee14_flows
import incidence
import loading_query
import numpy as np
import sparse_vs_dense
import two_libraries

## Two libraries, one job

pandapower stores results in `res_*` tables of a mutable `net`; pypowsybl updates the network in place and
exposes results as columns of `get_lines()`. Both are black boxes here: only the data handling matters.

In [ ]:
counts, ac_p, dc_p = two_libraries.pandapower_case57()
print('case57 (buses, lines, trafos):', counts)
print('largest |AC - DC| from-end flow difference:', round(float((ac_p - dc_p).abs().max()), 2), 'MW')
status, lines = two_libraries.pypowsybl_ieee14()
print('pypowsybl IEEE 14:', status)
lines.head()

## A type-annotated NumPy function

`incidence_matrix` builds the branch-bus incidence matrix with fancy indexing, no Python loop. Every row sums to
zero and $A^\top A$ is the graph Laplacian of Chapter 2.

In [ ]:
a = incidence.incidence_matrix([0, 0, 1, 1, 2], [1, 2, 2, 3, 3], 4)
print(a)
print('row sums:', a.sum(axis=1))
print(a.T @ a)

## Sparse against dense solves

Random sparse symmetric matrices, solved with `scipy.sparse.linalg.spsolve` and with `numpy.linalg.solve`.
The sizes stop at 800 here to keep the notebook quick; `python sparse_vs_dense.py` goes to 3200, where the
fill-in of the sparse LU on a random pattern lets the dense solver catch up.

In [ ]:
x_sparse, x_dense = sparse_vs_dense.solve_both(sparse_vs_dense.random_symmetric(200), np.ones(200))
print('max |x_sparse - x_dense| =', np.abs(x_sparse - x_dense).max())
for n, t_sparse, t_dense in sparse_vs_dense.timings((200, 400, 800), repeat=2):
    print(f'n = {n:4d}: spsolve {1e3 * t_sparse:7.2f} ms, dense {1e3 * t_dense:7.2f} ms')

## AC and DC line flows of IEEE 14 in pypowsybl

The frame is sorted by $|p_1|$ of the AC run; the bar chart puts the AC and DC series side by side.

In [ ]:
df14 = ieee14_flows.line_flows()
print(df14.round(2).head(6))
ieee14_flows.bar_chart(df14)

## case57: where the DC approximation is off

The lab's side-by-side table and histogram. Polars is not in the course environment, so `to_polars` is left
to the lab (run it with ToOp's interpreter).

In [ ]:
df57 = case57_ac_dc.side_by_side()
print(df57['difference_mw'].describe().round(2))
case57_ac_dc.histogram(df57)

## The same query in pandas (and in Polars, where installed)

Lines above the median loading, mean loading per from bus. The loadings of case57 are tiny because the
MATPOWER case ships no line ratings and pandapower fills `max_i_ka` with a very large value: the query, not the
percentages, is the point.

In [ ]:
res = loading_query.res_line_case57()
loading_query.pandas_query(res).sort_values(ascending=False).head()